In [ ]:
from huggingface_hub import login
login()

In [ ]:
from datasets import load_dataset, Dataset
import requests
import json
import pathlib
from collections import Counter
import re

In [ ]:
dataset="google/smol"
def list_smoldoc_configs(dataset="google/smol"):
    url = f"https://datasets-server.huggingface.co/splits?dataset={dataset}"
    r = requests.get(url); r.raise_for_status()
    data = r.json()
    configs = sorted({item["config"] for item in data["splits"]})
    return [c for c in configs if c.lower().startswith("smoldoc__")]

In [ ]:
smoldoc_configs = list_smoldoc_configs()
print(f"{len(smoldoc_configs)} SmolDoc configs")

In [ ]:
datasets_dict = {}
for cfg in smoldoc_configs:
    print("Loading config:", cfg)
    ds = load_dataset(dataset, cfg)
    datasets_dict[cfg] = ds

In [ ]:
# find all unique topic ids across all langauges
all_topic_ids = set()
for cfg, ds in datasets_dict.items():
    topic_ids = set(ds['train']['id'])
    all_topic_ids.update(topic_ids)
print(f"Total unique topic ids across all SmolDoc topics: {len(all_topic_ids)}")

In [ ]:
datasets_dict['smoldoc__en_bgq']['train']

In [ ]:
filtered_ds = datasets_dict['smoldoc__en_sw']['train'].filter(
    lambda example: example['factuality'] != 'ok'
)

# View the filtered dataset
print(f"Original dataset size: {len(datasets_dict['smoldoc__en_sw']['train'])}")
print(f"Filtered dataset size: {len(filtered_ds)}")

In [ ]:
filtered_ds

In [ ]:
from data_utils import load_topic_ratings
JSON_PATH = "smoldoc-factuality-ratings.json"  # adjust if needed


topic_ds = load_topic_ratings(JSON_PATH)
topic_ds

In [ ]:
annot_topic_ids = set([x for x in topic_ds["topic_key"] if x is not None])
english_cfgs = [cfg for cfg in datasets_dict.keys() if cfg.startswith("smoldoc__en_")]
english_topic_ids = set()
for cfg in english_cfgs:
    ds = datasets_dict[cfg]["train"]
    # Dataset stores 'id' as strings in HF; normalize to int
    english_topic_ids.update(x for x in ds["id"])

print(f"# Annotation rows (topic_ds): {len(topic_ds)}")
print(f"# Unique annotated topic_ids: {len(annot_topic_ids)}")
print(f"# English topic_ids in datasets: {len(english_topic_ids)}")

missing_in_annotations = sorted(english_topic_ids - annot_topic_ids)
extra_in_annotations_vs_english = sorted(annot_topic_ids - english_topic_ids)

print(f"# English topic_ids missing annotations: {len(missing_in_annotations)}")
if missing_in_annotations:
    print("Sample missing IDs:", missing_in_annotations[:20])

print(f"# Annotated topic_ids not present in English: {len(extra_in_annotations_vs_english)}")
if extra_in_annotations_vs_english:
    print("Sample extra IDs (annotated but not English):", extra_in_annotations_vs_english[:20])

In [ ]:
annot_fields_by_id = {}
keep_cols = ['annotator_1_label', 'annotator_1_notes',
             'annotator_2_label', 'annotator_2_notes',
             'annotator_3_label', 'annotator_3_notes']

for topic in topic_ds:
    topic_id = topic['topic_key']
    annot_fields_by_id[topic_id] = {k: topic[k] for k in keep_cols}

def _join_annotations(batch):
    annotation_columns = {k: [] for k in keep_cols}
    for topic_id in batch['id']:
        annots = annot_fields_by_id.get(topic_id, {})
        if annots:
            for k in keep_cols:
                annotation_columns[k].append(annots.get(k, None))
        else:
            for k in keep_cols:
                annotation_columns[k].append(None)
    data_dict = {**batch, **annotation_columns}
    return data_dict

fact_annot_ds = {}

for cfg, ds in datasets_dict.items():
    print("Processing config:", cfg)
    ds_with_annots = ds['train'].map(_join_annotations, batched=True)
    fact_annot_ds[cfg] = ds_with_annots

In [ ]:
example_cfg = 'smoldoc__en_sw'
fact_annot_ds[example_cfg]

In [ ]:
filtered_ids = set(filtered_ds['id'])

# Filter topic_ds to only include rows where topic_key is in filtered_ids
topic_ds_filtered = topic_ds.filter(
    lambda example: example['topic_key'] in filtered_ids
)
topic_ds_filtered
